Hands-on Task 3: Three-Way Conditional Router 

Objective 

Implement a routing system with multiple decision branches. 

Task 

Create a router that classifies user questions into: 

science 

history 

general 

Based on this classification, route the input to one of three nodes: 

science_node 

history_node 

general_node 

`` 

Node Behavior 

science_node → Respond as a science teacher 

history_node → Respond as a historian 

general_node → Respond as a general assistant 

Routing Requirement 

The routing function must return: 

Literal["science_node", "history_node", "general_node"] 

Testing Requirement 

Test with at least 4 questions, ensuring all categories are covered. 

Skills Tested 

Using add_conditional_edges 

Implementing multi-branch routing 

Applying Literal type hints 

Designing persona-based responses 

 

Expected Outcome 

By completing these tasks, you will be able to: 

Extend pipeline state using TypedDict 

Design multi-node workflows 

Build LLM-powered pipelines 

Route inputs dynamically across multiple branches 

Apply persona-based response strategies 

In [2]:
#Step 2: Imports
import os
from dotenv import load_dotenv
from typing_extensions import TypedDict, Literal
 
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic

print("setup complete")

setup complete


In [3]:
#Step 3: Create the LLM
 
# Load ANTHROPIC_API_KEY from .env
load_dotenv(override=True)
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
if not anthropic_api_key:
    raise ValueError("ANTHROPIC_API_KEY not found. Add it to your .env file.")

# ── Initialise Claude chat model ──────────────────────────────────────────
llm = ChatAnthropic(
    model=os.getenv("LLM_MODEL"),
    base_url=os.getenv("LLM_ENDPOINT"),
    temperature=0
)
 
print("llm created!")

llm created!


In [5]:
#Step 4: Define State
class RouterState(TypedDict):
    question: str
    category: str
    answer: str

In [7]:
#Step 5: Router Function
def router(state: RouterState) -> Literal["science_node", "history_node", "general_node"]:
 
    question = state["question"].lower()
 
    science_keywords = [
        "science","physics","chemistry","biology",
        "planet","gravity","atom","energy"
    ]
 
    history_keywords = [
        "history","war","king","queen","empire",
        "independence","freedom","ashoka"
    ]
 
    if any(word in question for word in science_keywords):
        state["category"] = "Science"
        return "science_node"
 
    elif any(word in question for word in history_keywords):
        state["category"] = "History"
        return "history_node"

    else:
        state["category"] = "General"
        return "general_node"
 

In [21]:
#Step 6: Science Node
def science_node(state: RouterState):
 
    prompt = f"""
    You are a science teacher.
 
    Answer the following question clearly.
 
    Question:
    {state['question']}
    """
 
    response = llm.invoke(prompt)
 
    return {
        "category":"Science",
        "answer": response.content
        }

In [22]:
#Step 7: History Node
def history_node(state: RouterState):
 
    prompt = f"""
    You are a historian.
 
    Answer this question.
 
    Question:
    {state['question']}
    """
 
    response = llm.invoke(prompt)
 
    return {
        "category":"History",
        "answer": response.content}


In [23]:
#Step 8: General Node
def general_node(state: RouterState):
 
    prompt = f"""
    You are a helpful AI assistant.
 
    Answer the question.
 
    Question:
    {state['question']}
    """
 
    response = llm.invoke(prompt)
 
    return {
        "category":"General",
        "answer": response.content}

#Step 9: Build Graph

In [24]:

builder = StateGraph(RouterState)
 
builder.add_node("science_node", science_node)
builder.add_node("history_node", history_node)
builder.add_node("general_node", general_node)

#Step 10: Add Conditional Routing
#This is the important part of today's task.

In [25]:

builder.add_conditional_edges(
    START,
    router
)

In [26]:
#Step 11: Connect to END
builder.add_edge("science_node", END)
builder.add_edge("history_node", END)
builder.add_edge("general_node", END)

In [27]:
#Step 12: Compile
graph = builder.compile()

#Step 13: Test (at least 4 questions)

In [30]:

#Test 1
result = graph.invoke({
    "question": "Why is the sky blue?"
})

print(result["category"])
print(result["answer"])

General
# Why is the Sky Blue?

The sky appears blue due to a phenomenon called **Rayleigh scattering**.

## How it works:

1. **Sunlight enters the atmosphere** - The sun emits white light containing all colors of the visible spectrum.

2. **Light scatters off air molecules** - When sunlight hits nitrogen and oxygen molecules in the atmosphere, it scatters in all directions.

3. **Blue light scatters more** - Blue light has a shorter wavelength than other colors (like red or yellow). Shorter wavelengths scatter much more effectively than longer wavelengths—about 10 times more than red light.

4. **Our eyes see the scattered blue** - Because blue light is scattered so much, it reaches our eyes from all directions across the sky, making the entire sky appear blue.

## Why not violet?

You might wonder why the sky isn't violet, since violet has an even shorter wavelength. This is because:
- The sun emits less violet light than blue light
- Violet light is absorbed more by the upper atmos

In [31]:
#Test 2
result = graph.invoke({
    "question": "Who was Ashoka?"
})
 
print(result["category"])
print(result["answer"])

History
# Ashoka: The Great Mauryan Emperor

**Ashoka** (c. 304–232 BCE) was one of ancient India's most significant rulers and the third emperor of the Mauryan Empire.

## Key Aspects of His Life and Legacy:

**Early Reign**
- Known initially as a fierce military conqueror who expanded the empire through aggressive campaigns
- His conquest of Kalinga (modern-day Odisha) around 260 BCE was particularly brutal, killing an estimated 100,000 people

**Spiritual Transformation**
- Deeply troubled by the violence and suffering caused by his conquests, Ashoka underwent a profound spiritual conversion
- Embraced Buddhism and adopted a philosophy of non-violence (*ahimsa*)
- This transformation became a turning point in his reign and legacy

**Achievements**
- Spread Buddhism throughout Asia, becoming one of history's greatest missionary rulers
- Erected stone pillars and edicts across his empire to promote Buddhist teachings and moral governance
- Established hospitals, schools, and rest hous

In [32]:
#Test 3
result = graph.invoke({
    "question": "Tell me a joke."
})
 
print(result["category"])
print(result["answer"])

General
# Here's one for you:

Why don't scientists trust atoms?

Because they make up everything! 😄


In [34]:
#Test 4
result = graph.invoke({
    "question": "Explain gravity."
})

print(result["category"])
print(result["answer"])

Science
# Gravity Explained

## What is Gravity?

Gravity is a **fundamental force of nature** that attracts objects with mass toward each other. It's the force that keeps us on Earth and holds planets in orbit around the Sun.

## Key Points:

### The Basics
- Every object with mass has gravity
- Gravity pulls objects *toward* each other
- The larger the mass, the stronger the gravitational pull

### Why We Experience Gravity
- Earth's mass is enormous, so its gravity is very strong
- This force pulls us downward (toward Earth's center)
- It gives us our **weight**

### How Gravity Works at a Distance
- Gravity doesn't require objects to touch
- It acts invisibly through space
- The farther apart objects are, the weaker the gravitational force

## Real-World Examples:
- 🍎 An apple falls from a tree
- 🌍 We stay on the ground instead of floating away
- 🌙 The Moon orbits Earth
- 🪐 Planets orbit the Sun

## Important Note:
While we experience gravity every day, scientists still don't fully

# Task 3: LangGraph Structure (Three-Way Conditional Router)
 
```text
                         START
                           │
                           ▼
                    router_node
                           │
        ┌──────────────────┼──────────────────┐
        │                  │                  │
        ▼                  ▼                  ▼
 science_node       history_node      general_node
        │                  │                  │
        │                  │                  │
        └──────────────────┼──────────────────┘
                           ▼
                          END
```
 
### Flow
 
1. **START**
   - User enters a question.
 
2. **router_node**
   - Classifies the question into one of three categories:
     - Science
     - History
     - General
 
3. **Conditional Routing**
   - Science → `science_node`
   - History → `history_node`
   - General → `general_node`
 
4. **Response Nodes**
   - **science_node** → Respond like a science teacher.
   - **history_node** → Respond like a historian.
   - **general_node** → Respond like a general assistant.
 
5. **END**
   - Return the final response.
```
 